# 01. Validation grid search

## Setup

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from experiments.tuning import tuning
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

## Inputs

In [ ]:
development_path = ""  # Optional features_development.json; blank uses preprocessing 03
n_estimators = 500
cohorts = list(tuning.COHORTS)
features_path, development = tuning.load_development(root, development_path or None)
print("Development features:", features_path)
for cohort in cohorts:
    train, validation = tuning.cohort_split(development, cohort)
    print(f"{cohort}: train {len(train)} pixels / {len({r['partition_group'] for r in train})} parcels, "
          f"validation {len(validation)} pixels / {len({r['partition_group'] for r in validation})} parcels")

## Search space

In [ ]:
search_space = dict(tuning.DEFAULT_SPACE)
grid = tuning.parameter_grid(search_space, n_estimators=n_estimators)
print("Settings per cohort:", len(grid))
search_space

## Run search

In [ ]:
run = tuning.new_run(root, "grid", features_path, n_estimators=n_estimators,
                     search_space={k: [str(v) for v in values] for k, values in search_space.items()})
records = tuning.validation_search(development, grid, cohorts=cohorts,
                                   progress=lambda message: print(message, flush=True))
print("Saved:", tuning.save_records(run, "validation_grid", records))

## Results table

In [ ]:
results = pd.DataFrame(records)
results["max_depth"] = results["max_depth"].astype(str)
results["max_samples"] = results["max_samples"].astype(str)
for cohort in cohorts:
    print(cohort)
    display(results[results.cohort == cohort].sort_values(["macro_f1", "balanced_accuracy"], ascending=False).head(10)
            .drop(columns=["cohort", "seed"]).round(3))

## Baseline settings compared with best setting

In [ ]:
baseline = results[(results.max_features == "sqrt") & (results.max_samples == "0.5")
                   & (results.max_depth == "None") & results.min_samples_leaf.isin([2, 4])]
best = results.loc[results.groupby("cohort").macro_f1.idxmax()]
comparison = pd.concat([baseline.assign(row="baseline"), best.assign(row="best")])
display(comparison.sort_values(["cohort", "row"])[["cohort", "row", "max_features", "min_samples_leaf",
        "max_samples", "max_depth", "macro_f1", "balanced_accuracy", "carrot_f1", "cabbage_f1", "potato_f1"]].round(3))

## Parameter effects

In [ ]:
parameters = ["max_features", "min_samples_leaf", "max_samples", "max_depth"]
figure, axes = plt.subplots(1, len(parameters), figsize=(18, 4), sharey=True)
for axis, parameter in zip(axes, parameters):
    table = results.assign(**{parameter: results[parameter].astype(str)}).groupby([parameter, "cohort"]).macro_f1.mean().unstack()
    table.plot(kind="bar", ax=axis, legend=axis is axes[-1])
    axis.set_title(parameter)
    axis.set_ylabel("Mean validation parcel macro-F1")
plt.tight_layout()
figure.savefig(run / "parameter_effects.png", dpi=150)
display(figure)
plt.close(figure)

## Heatmaps

In [ ]:
figure, axes = plt.subplots(1, len(cohorts), figsize=(5 * len(cohorts), 4))
for axis, cohort in zip(axes, cohorts):
    table = results[results.cohort == cohort].pivot_table(index="max_features", columns="min_samples_leaf",
                                                          values="macro_f1", aggfunc="max")
    image = axis.imshow(table.values, vmin=0, vmax=1, cmap="viridis")
    axis.set_xticks(range(len(table.columns)), table.columns)
    axis.set_yticks(range(len(table.index)), [str(v) for v in table.index])
    for (i, j), value in pd.DataFrame(table.values).stack().items():
        axis.text(j, i, f"{value:.2f}", ha="center", va="center", color="white")
    axis.set_title(cohort)
    axis.set_xlabel("min_samples_leaf")
    axis.set_ylabel("max_features")
plt.tight_layout()
figure.savefig(run / "heatmaps.png", dpi=150)
display(figure)
plt.close(figure)
print("Run folder:", run)